**Cell #01**

# RAG11 Nutrition — Stage 1.2: Load Chunks to Supabase

Reads every `stage1_eda_output/sources/source_row-N.json` file plus every
`parent_chunk-N.json` / `child_chunk-parentN-chunkM.json` file written by
`stage1_1_extract_and_chunk.ipynb`, embeds each child chunk's text with
Voyage AI, and upserts rows into `rag11_data_sources`,
`rag11_chunks_parent_table`, and `rag11_chunks_child_table` (in that order --
parent/child rows carry a foreign key onto `rag11_data_sources`).

**Before running this notebook**: `sql/create_sql_tables.sql` must already have
been run once in the Supabase Dashboard -> SQL Editor (tables, HNSW index,
and RPC functions).

Row shape matches that schema exactly — `rowGUID` / `rowOwnerGUID` /
`rowParentGUID` / `orderInList` / `rowJSON`:
- `rowGUID` — deterministic `uuid5` derived from a stable business key (a
  source's Drive `file_id`, or a chunk's own `parent_id`/`child_id`), so
  re-running this notebook **upserts** existing rows instead of duplicating
  them (same idempotent-ingestion pattern as `poc_pipline_ipynb.ipynb`).
- `rowOwnerGUID` — the owning source's `rag11_data_sources` row: its own
  `rowGUID` for a source row (self-owned), or that source's `rowGUID` for
  every parent/child row produced from it (read from each chunk file's
  `source_row_guid` field, which `stage1_1_extract_and_chunk.ipynb` writes).
  This replaced the earlier plain-text `'source1'`/`'source2'` key.
- `rowParentGUID` — `None` for a source row (nothing sits above it); on a
  parent-chunk row, `None`; on a child row, the deterministic `uuid5`
  computed from the parent's `parent_id`, so it always matches the parent
  row's `rowGUID` and satisfies the foreign key.
- `orderInList` — the numeric index parsed out of the chunk's file name (or
  the source's position in the Drive folder listing, for source rows).
- `rowJSON` — the source/chunk's JSON file content, verbatim.


In [1]:
# Cell #02
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


**Cell #03**

## Imports & client setup

In [2]:
# Cell #04
import os
import re
import json
import time
import uuid
import random
import resource
import concurrent.futures as cf
from pathlib import Path
from typing import Optional

from dotenv import load_dotenv
from supabase import create_client, Client
from tqdm.auto import tqdm
import voyageai

load_dotenv()


def require_env(name: str) -> str:
    """Fetch an env var and fail with a clear, actionable message (naming the
    exact .env line to fill in) instead of a cryptic downstream traceback
    like SupabaseException('supabase_key is required')."""
    value = os.environ.get(name, "").strip()
    if not value:
        raise RuntimeError(
            f"{name} is empty in your .env file. Open .env in the RAG11 folder "
            f"and paste your actual value in after '{name}='."
        )
    return value


SUPABASE_URL = require_env("PUBLIC_SUPABASE_URL")
# Prefer the service_role key (bypasses RLS cleanly); falls back to the anon
# key, which only works with the permissive policies create_sql_tables.sql
# already sets up.
SUPABASE_KEY = os.environ.get("SUPABASE_SERVICE_ROLE_KEY", "").strip() or require_env("PUBLIC_SUPABASE_ANON_KEY")
VOYAGE_API_KEY = require_env("VOYAGE_API_KEY")

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
voyage_client = voyageai.Client(api_key=VOYAGE_API_KEY)

OUTPUT_ROOT = Path(".") / "stage1_eda_output"
CHECKPOINT_DIR = OUTPUT_ROOT / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
SOURCES_MANIFEST_DIR = OUTPUT_ROOT / "sources"
# Discovered from disk instead of hardcoded, so a source4/5/... that
# stage1_1_extract_and_chunk.ipynb picked up from Drive is loaded here
# too without editing this notebook.
SOURCE_KEYS = sorted(
    (p.name for p in OUTPUT_ROOT.iterdir() if p.is_dir() and re.match(r"^source\d+$", p.name)),
    key=lambda k: int(k[len("source"):]),
)
SOURCES_TABLE = "rag11_data_sources"
PARENT_TABLE = "rag11_chunks_parent_table"
CHILD_TABLE = "rag11_chunks_child_table"
EMBEDDING_MODEL = "voyage-3"   # 1024-dimensional -- must match vector(...) width in create_sql_tables.sql
EMBEDDING_DIM = 1024

soft_fd_limit, hard_fd_limit = resource.getrlimit(resource.RLIMIT_NOFILE)
print(f"This kernel process's open-file limit: soft={soft_fd_limit}, hard={hard_fd_limit}")
if soft_fd_limit < 2048:
    print("  -> LOW. If you see repeated '[Errno 35] Resource temporarily unavailable' "
          "during upserts, this is almost certainly why: this Jupyter/PyCharm process "
          "was started before stage1_0_run_mac_settings.command took effect. Fully quit "
          "and reopen Jupyter/PyCharm (a re-run of this cell alone won't pick up a new "
          "system limit -- the whole process needs to restart), then re-run.")

print("Clients ready. Supabase project:", SUPABASE_URL)


This kernel process's open-file limit: soft=1048576, hard=9223372036854775807
Clients ready. Supabase project: https://czgrxgzdmodkkmbmraub.supabase.co


**Cell #05**

## Load chunk JSON files

Parses the `N` / `M` indices straight out of each file name (rather than
trusting `parent_id`/`child_id` to always be well-formed) so `orderInList`
is always a genuine integer position within its source/parent.

In [3]:
# Cell #06
PARENT_FILE_RE = re.compile(r"^parent_chunk-(\d+)\.json$")
CHILD_FILE_RE = re.compile(r"^child_chunk-parent(\d+)-chunk(\d+)\.json$")


def load_parent_files(source_key: str) -> list[tuple[int, dict]]:
    """Return [(orderInList, parent_json_dict), ...] sorted by order."""
    out = []
    folder = OUTPUT_ROOT / source_key
    for f in folder.glob("parent_chunk-*.json"):
        m = PARENT_FILE_RE.match(f.name)
        if not m:
            continue
        order = int(m.group(1))
        out.append((order, json.loads(f.read_text(encoding="utf-8"))))
    out.sort(key=lambda t: t[0])
    return out


def load_child_files(source_key: str) -> list[tuple[int, int, dict]]:
    """Return [(parent_order, child_order, child_json_dict), ...] sorted."""
    out = []
    folder = OUTPUT_ROOT / source_key
    for f in folder.glob("child_chunk-*.json"):
        m = CHILD_FILE_RE.match(f.name)
        if not m:
            continue
        p_order, c_order = int(m.group(1)), int(m.group(2))
        out.append((p_order, c_order, json.loads(f.read_text(encoding="utf-8"))))
    out.sort(key=lambda t: (t[0], t[1]))
    return out


parents_by_source = {k: load_parent_files(k) for k in SOURCE_KEYS}
children_by_source = {k: load_child_files(k) for k in SOURCE_KEYS}

for k in SOURCE_KEYS:
    print(f"[{k}] {len(parents_by_source[k])} parent file(s), {len(children_by_source[k])} child file(s)")


[source1] 133 parent file(s), 8 child file(s)
[source2] 136 parent file(s), 184 child file(s)
[source3] 91 parent file(s), 185 child file(s)
[source4] 15 parent file(s), 46 child file(s)
[source5] 22 parent file(s), 106 child file(s)
[source6] 22 parent file(s), 106 child file(s)
[source7] 94 parent file(s), 75 child file(s)
[source8] 97 parent file(s), 294 child file(s)
[source9] 35 parent file(s), 173 child file(s)
[source10] 9 parent file(s), 101 child file(s)
[source11] 30 parent file(s), 712 child file(s)
[source12] 15 parent file(s), 125 child file(s)
[source13] 28 parent file(s), 195 child file(s)
[source14] 18 parent file(s), 123 child file(s)
[source15] 42 parent file(s), 188 child file(s)
[source16] 245 parent file(s), 300 child file(s)
[source17] 15 parent file(s), 192 child file(s)


**Cell #07**

## Deterministic row IDs

Same trick as `poc_pipline_ipynb.ipynb`: derive every `rowGUID` from a
stable business key (the chunk's own `parent_id`/`child_id` string) via
`uuid5`, so re-running this notebook is a safe upsert rather than a
duplicate insert or an orphaned foreign key.

In [4]:
# Cell #08
RAG11_UUID_NAMESPACE = uuid.uuid5(uuid.NAMESPACE_DNS, "rag11.nutrition.poc")


def deterministic_uuid(business_key: str) -> str:
    return str(uuid.uuid5(RAG11_UUID_NAMESPACE, business_key))


**Cell #09**

## Voyage AI embeddings — parallel, batched, retried

Identical pattern to `poc_pipline_ipynb.ipynb`'s embedding helper.

In [5]:
# Cell #10
# Postgrest errors whose cause is the *data*, not the network -- retrying
# these just burns ~2 extra minutes (8 attempts of exponential backoff)
# before failing anyway with the exact same error. Fail fast instead.
_NON_RETRYABLE_POSTGREST_CODES = {
    "22P05",  # unsupported Unicode escape sequence -- e.g. a literal
              # \u0000 inside a text/jsonb column. Postgres's text type
              # can never store \u0000 (it's C-string-backed under the
              # hood), so no amount of retrying fixes this -- the row
              # itself needs the NUL byte stripped before it's sent. See
              # _strip_bad_unicode / _sanitize_row_for_upsert below.
}


def _is_non_retryable_postgrest_error(e: Exception) -> bool:
    return getattr(e, "code", None) in _NON_RETRYABLE_POSTGREST_CODES


def _retry(fn, *args, max_attempts: int = 8, base_delay: float = 2.0, **kwargs):
    """Exponential-backoff retry wrapper for flaky network calls."""
    last_exc = None
    for attempt in range(1, max_attempts + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_exc = e
            if _is_non_retryable_postgrest_error(e):
                print(f"[fatal] {getattr(fn, '__name__', fn)} hit a non-retryable "
                      f"Postgrest error ({e}); not retrying -- this is a data "
                      f"problem, not a network blip.")
                raise
            if attempt == max_attempts:
                break
            sleep_s = base_delay * (2 ** (attempt - 1)) + random.uniform(0, 0.5)
            print(f"[retry] {getattr(fn, '__name__', fn)} attempt {attempt} "
                  f"failed ({e}); retrying in {sleep_s:.1f}s")
            time.sleep(sleep_s)
    raise last_exc


def _batched(seq: list, size: int):
    """Yield consecutive chunks of `seq`, each of length <= size."""
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


VOYAGE_BATCH_SIZE = 64      # texts per embed() call (well under Voyage's per-request cap)
VOYAGE_MAX_WORKERS = 4      # concurrent in-flight embedding requests


def embed_texts(texts: list[str], input_type: str = "document") -> list[list[float]]:
    """
    Embed `texts` with Voyage AI in parallel batches, preserving input order.
    """
    batches = list(_batched(texts, VOYAGE_BATCH_SIZE))

    def _embed_one_batch(batch: list[str]) -> list[list[float]]:
        resp = _retry(
            voyage_client.embed,
            texts=batch,
            model=EMBEDDING_MODEL,
            input_type=input_type,
        )
        return resp.embeddings

    results: list[Optional[list[list[float]]]] = [None] * len(batches)
    with cf.ThreadPoolExecutor(max_workers=VOYAGE_MAX_WORKERS) as pool:
        future_to_idx = {pool.submit(_embed_one_batch, b): i for i, b in enumerate(batches)}
        with tqdm(total=len(batches), desc="Embedding batches", unit="batch") as bar:
            for future in cf.as_completed(future_to_idx):
                idx = future_to_idx[future]
                results[idx] = future.result()
                bar.update(1)

    flattened: list[list[float]] = []
    for batch_embeddings in results:
        flattened.extend(batch_embeddings)
    return flattened


**Cell #11**

## Batched + parallel Supabase upserts

Same helper as `poc_pipline_ipynb.ipynb`. Upsert (not plain insert) is what
makes re-running this notebook safe.

Also **checkpointed**: every successfully-upserted `rowGUID` is recorded to
`stage1_eda_output/_checkpoints/<table>_upserted_row_guids.json`, and a
re-run skips rows already checkpointed. So if a big load dies partway
through (network blip, a resource limit, closing the laptop), re-running
the notebook only pushes the rows that didn't make it, instead of
re-upserting everything from scratch.

**If you truncate the Supabase tables** (`delete_chunks_data.sql`), also
delete the `stage1_eda_output/_checkpoints/` folder — otherwise this
notebook will think those rows are already loaded and skip them, even
though the tables are now empty.

In [6]:
# Cell #12
# [Errno 35] Resource temporarily unavailable during upserts, recurring even
# at low concurrency, points to accumulated open-file-descriptor pressure
# (short-lived connections sitting in TCP TIME_WAIT after each request) on
# a process whose open-file limit is still at macOS's low default --
# see the "open-file limit" print in the client-setup cell above; the real
# fix is restarting Jupyter/PyCharm after stage1_0_run_mac_settings.command.
# Dropped to fully sequential (1 worker) as a safe fallback in the
# meantime -- slower, but avoids the failure entirely regardless of the
# process's file-descriptor headroom.
SUPABASE_INSERT_BATCH_SIZE = 100   # rows per upsert() call
SUPABASE_MAX_WORKERS = 1           # sequential -- raise back to 2-3 once the fd limit above reads >= 2048
SUPABASE_SUBMIT_STAGGER_S = 0.3    # delay between launching each worker


def _checkpoint_path(table_name: str) -> Path:
    return CHECKPOINT_DIR / f"{table_name}_upserted_row_guids.json"


def _load_checkpoint(table_name: str) -> set[str]:
    p = _checkpoint_path(table_name)
    if p.exists():
        return set(json.loads(p.read_text(encoding="utf-8")))
    return set()


def _save_checkpoint(table_name: str, row_guids: set[str]) -> None:
    _checkpoint_path(table_name).write_text(
        json.dumps(sorted(row_guids)), encoding="utf-8"
    )


def _strip_bad_unicode(value):
    """Recursively strip NUL bytes (\u0000) out of every string nested
    inside `value` (dicts/lists/strings), leaving other types untouched.

    Postgres text/jsonb columns cannot store \u0000 at all -- it's not a
    formatting quirk, the on-disk text type is C-string-backed, so a NUL
    truncates it. This is what upsert() was choking on:
    APIError 22P05 'unsupported Unicode escape sequence' /
    '\\u0000 cannot be converted to text.' Source PDFs/HTML occasionally
    contain a stray NUL from bad encoding/OCR, which then rides along
    inside rowJSON. Stripping it here (rather than upstream in every
    build_*_row function) means every table that goes through
    _upsert_batches is protected, including any future one.
    """
    if isinstance(value, str):
        return value.replace("\x00", "")
    if isinstance(value, dict):
        return {k: _strip_bad_unicode(v) for k, v in value.items()}
    if isinstance(value, list):
        return [_strip_bad_unicode(v) for v in value]
    return value


def _sanitize_row_for_upsert(row: dict) -> dict:
    """Sanitize one row dict (source/parent/child) right before it's sent."""
    return _strip_bad_unicode(row)


def _upsert_batches(table_name: str, rows: list[dict],
                     batch_size: int = SUPABASE_INSERT_BATCH_SIZE,
                     max_workers: int = SUPABASE_MAX_WORKERS) -> int:
    already_done = _load_checkpoint(table_name)
    remaining_rows = [r for r in rows if r["rowGUID"] not in already_done]
    skipped = len(rows) - len(remaining_rows)
    if skipped:
        print(f"  [{table_name}] resuming: skipping {skipped} row(s) "
              f"already upserted in a previous run")

    batches = list(_batched(remaining_rows, batch_size))
    inserted = 0

    def _upsert_one_batch(batch: list[dict]) -> list[str]:
        clean_batch = [_sanitize_row_for_upsert(r) for r in batch]
        _retry(lambda: supabase.table(table_name).upsert(clean_batch).execute())
        return [r["rowGUID"] for r in batch]

    with cf.ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = []
        for b in batches:
            futures.append(pool.submit(_upsert_one_batch, b))
            time.sleep(SUPABASE_SUBMIT_STAGGER_S)
        with tqdm(total=len(batches), desc=f"Upserting {table_name}", unit="batch") as bar:
            for future in cf.as_completed(futures):
                guids = future.result()
                inserted += len(guids)
                already_done.update(guids)
                # Checkpoint after every completed batch (from this single
                # main thread only, so there's no concurrent-write race) --
                # so a crash mid-run loses at most the in-flight batches.
                _save_checkpoint(table_name, already_done)
                bar.update(1)

    return inserted


**Cell #13**

## Load & upsert sources -- `rag11_data_sources`

Loaded and upserted *before* parent rows, since
`rag11_chunks_parent_table.rowOwnerGUID` (and the child table's) carries a
foreign key onto `rag11_data_sources.rowGUID`.


In [7]:
# Cell #14
def load_source_rows() -> list[dict]:
    return [
        json.loads(f.read_text(encoding="utf-8"))
        for f in sorted(SOURCES_MANIFEST_DIR.glob("source_row-*.json"))
    ]


source_rows = load_source_rows()
if not source_rows:
    raise RuntimeError(
        f"No source manifest files found in {SOURCES_MANIFEST_DIR} -- run "
        "stage1_1_extract_and_chunk.ipynb first."
    )

# source_key -> its rag11_data_sources rowGUID. Used below as a fallback
# for chunk files written before source_row_guid was added to their JSON.
source_row_guid_by_key = {r["rowJSON"]["source_key"]: r["rowGUID"] for r in source_rows}

print(f"Upserting {len(source_rows)} source row(s) into {SOURCES_TABLE}...")
_t0 = time.monotonic()
n_sources = _upsert_batches(SOURCES_TABLE, source_rows)
_elapsed = time.monotonic() - _t0
print(f"  -> {n_sources} source row(s) upserted in {_elapsed:.1f}s.")


Upserting 17 source row(s) into rag11_data_sources...
  [rag11_data_sources] resuming: skipping 17 row(s) already upserted in a previous run


Upserting rag11_data_sources: 0batch [00:00, ?batch/s]

  -> 0 source row(s) upserted in 0.0s.


**Cell #15**

## Build & upsert parent rows

Parents are upserted *before* children, since the child table's
`rowParentGUID` has a foreign key onto the parent table.

In [8]:
# Cell #16
def build_parent_row(source_key: str, order: int, data: dict) -> dict:
    owner_guid = data.get("source_row_guid") or source_row_guid_by_key.get(source_key)
    if not owner_guid:
        raise RuntimeError(
            f"No rag11_data_sources rowGUID found for '{source_key}' -- "
            "re-run stage1_1_extract_and_chunk.ipynb (it writes "
            "source_row_guid into every parent/child chunk file, and a "
            "source_row-N.json manifest per source)."
        )
    return {
        "rowGUID": deterministic_uuid(f"parent:{data['parent_id']}"),
        "rowOwnerGUID": owner_guid,
        "rowParentGUID": None,
        "orderInList": order,
        "rowJSON": data,
    }


parent_rows = [
    build_parent_row(source_key, order, data)
    for source_key in SOURCE_KEYS
    for order, data in parents_by_source[source_key]
]

print(f"Upserting {len(parent_rows)} parent rows into {PARENT_TABLE}...")
_t0 = time.monotonic()
n_parents = _upsert_batches(PARENT_TABLE, parent_rows)
_elapsed = time.monotonic() - _t0
_rate = n_parents / _elapsed if _elapsed > 0 else float("inf")
print(f"  -> {n_parents} parent rows upserted in {_elapsed:.1f}s ({_rate:.1f} rows/s).")


Upserting 1047 parent rows into rag11_chunks_parent_table...
  [rag11_chunks_parent_table] resuming: skipping 300 row(s) already upserted in a previous run


Upserting rag11_chunks_parent_table:   0%|          | 0/8 [00:00<?, ?batch/s]

  -> 747 parent rows upserted in 10.2s (73.1 rows/s).


**Cell #17**

## Embed, build & upsert child rows

All child chunks across all three sources are embedded in one combined
sweep (keeps every Voyage batch full-sized instead of capping out per
source), then mapped onto rows and upserted.

In [9]:
# Cell #18
all_children = [
    (source_key, c_order, data)
    for source_key in SOURCE_KEYS
    for _p_order, c_order, data in children_by_source[source_key]
]

texts = [data["text"] for _source_key, _order, data in all_children]
print(f"Embedding {len(texts)} child chunks "
      f"(batch size {VOYAGE_BATCH_SIZE}, {VOYAGE_MAX_WORKERS} parallel workers)...")
_t0 = time.monotonic()
embeddings = embed_texts(texts, input_type="document")
_elapsed = time.monotonic() - _t0
print(f"  -> embedded {len(embeddings)} chunks in {_elapsed:.1f}s "
      f"({len(embeddings) / _elapsed if _elapsed > 0 else float('inf'):.1f} chunks/s).")
assert len(embeddings) == len(all_children), "embedding count mismatch"


def build_child_row(source_key: str, order: int, data: dict, embedding: list[float]) -> dict:
    owner_guid = data.get("source_row_guid") or source_row_guid_by_key.get(source_key)
    if not owner_guid:
        raise RuntimeError(
            f"No rag11_data_sources rowGUID found for '{source_key}' -- "
            "re-run stage1_1_extract_and_chunk.ipynb (it writes "
            "source_row_guid into every parent/child chunk file, and a "
            "source_row-N.json manifest per source)."
        )
    return {
        "rowGUID": deterministic_uuid(f"child:{data['child_id']}"),
        "rowOwnerGUID": owner_guid,
        "rowParentGUID": deterministic_uuid(f"parent:{data['parent_id']}"),
        "orderInList": order,
        "rowJSON": data,
        "embedding": embedding,
    }


child_rows = [
    build_child_row(source_key, order, data, embedding)
    for (source_key, order, data), embedding in zip(all_children, embeddings)
]

print(f"Upserting {len(child_rows)} child rows into {CHILD_TABLE}...")
_t0 = time.monotonic()
n_children = _upsert_batches(CHILD_TABLE, child_rows)
_elapsed = time.monotonic() - _t0
_rate = n_children / _elapsed if _elapsed > 0 else float("inf")
print(f"  -> {n_children} child rows upserted in {_elapsed:.1f}s ({_rate:.1f} rows/s).")

print("Stage 1.2 ingestion complete.")


Embedding 3113 child chunks (batch size 64, 4 parallel workers)...


Embedding batches:   0%|          | 0/49 [00:00<?, ?batch/s]

  -> embedded 3113 chunks in 91.8s (33.9 chunks/s).
Upserting 3113 child rows into rag11_chunks_child_table...


Upserting rag11_chunks_child_table:   0%|          | 0/32 [00:00<?, ?batch/s]

  -> 3113 child rows upserted in 101.3s (30.7 rows/s).
Stage 1.2 ingestion complete.
